# ⚡ Streaming & Event-Queue — der Agentic Loop in Echtzeit

**Derselbe Loop wie immer — aber er *streamt*, und er holt sich seine Aufträge selbst.**

Bisher war unser `run_agent` **blockierend**: Man sieht erst etwas, wenn ein Schritt
fertig ist (per `print`), und der Agent verarbeitet genau *einen* Auftrag pro Aufruf.
Das ist für Tests okay — aber für eine echte Anwendung fühlt es sich tot an.

Zwei Bausteine ändern das, ohne den Loop neu zu erfinden:

> **Streaming** — das Modell liefert seine Antwort *Token für Token*. Der Text erscheint
> sofort und wächst live (wie bei ChatGPT/Claude). Das senkt die gefühlte Wartezeit
> drastisch (*time-to-first-token*).
>
> **Event-Queue** — statt direkt zu `print`en, *publiziert* der Loop typisierte
> **Events** (Token, Tool-Call, Tool-Ergebnis, finale Antwort). Ein **Auftrags-Queue**
> füttert den Agenten als **Worker**; ein **Event-Bus** verteilt den Strom an *mehrere*
> Consumer (UI + Metriken) gleichzeitig.

Kernidee: Wir entkoppeln **„was passiert"** (der Loop) von **„wie es angezeigt wird"**
(die Consumer). Kein Framework, nur ein Producer/Consumer-Muster um denselben Loop.

| Kapitel | Frage | Zutat |
|---|---|---|
| 0 | Womit reden wir? | Setup + `stream_llm()` |
| 1 | Warum überhaupt streamen? | `stream=True`, Token-Deltas |
| 2 | Wie kommen Tools durch den Stream? | Deltas zusammensetzen |
| 3 | Wie werden `print`s zu Events? | `AgentEvent` |
| 4 | Wer reicht die Events herum? | Auftrags-Queue + Event-Bus |
| 5 | Wie sieht der Loop jetzt aus? | `run_agent_streaming()` |
| 6 | Alles zusammen, live | Worker + 2 Consumer |

## 0 · Setup

Wie in den anderen Notebooks kommt die Konfiguration komplett aus der `.env`
(Vorlage: `.env.example`). Neu ist nur `stream_llm()` — derselbe Call wie `llm()`,
aber mit `stream=True`, sodass wir die Antwort in **Chunks** statt am Stück bekommen.

In [ ]:
import os, json, time, queue, threading
from dataclasses import dataclass, field
from typing import Any
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()  # liest die .env

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
)
DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]


def llm(messages, tools=None, tool_choice="auto"):
    """Klassischer Call: Nachrichten rein -> EINE fertige Antwort raus."""
    kwargs = dict(model=DEPLOYMENT, messages=messages)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs)


def stream_llm(messages, tools=None, tool_choice="auto"):
    """Derselbe Call mit stream=True: liefert einen Iterator über CHUNKS.
    Jeder Chunk trägt nur ein winziges Delta (ein paar Zeichen Text oder ein
    Stück eines tool_calls)."""
    kwargs = dict(model=DEPLOYMENT, messages=messages, stream=True)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs)


print("Client bereit. Deployment:", DEPLOYMENT)

### Tool-Registry & Demo-Tools

Dieselbe winzige Registry wie im Grund-Notebook: ein `@register`-Dekorator pflegt
`TOOLS` (JSON-Schemas fürs Modell) und `DISPATCH` (Name → Python-Funktion).

Damit Streaming *und* Tool-Events im Vortrag sichtbar werden, geben wir zwei Tools
eine **simulierte Latenz** (`time.sleep`) — so sieht man live, wie der Agent „arbeitet".

In [ ]:
TOOLS = []        # JSON-Schemas fürs Modell
DISPATCH = {}     # name -> Python-Funktion


def register(name, description, parameters):
    def deco(fn):
        TOOLS.append({"type": "function", "function": {
            "name": name, "description": description, "parameters": parameters}})
        DISPATCH[name] = fn
        return fn
    return deco


@register("list_files", "Listet die Dateien in einem Verzeichnis auf.",
          {"type": "object", "properties": {"path": {"type": "string"}}, "required": []})
def _list_files(path="."):
    return "\n".join(sorted(os.listdir(path)))


@register("wetter", "Liefert das aktuelle Wetter für eine Stadt (simuliert).",
          {"type": "object", "properties": {"stadt": {"type": "string"}},
           "required": ["stadt"]})
def _wetter(stadt):
    time.sleep(1.2)                       # simulierte Netz-Latenz
    return f"{stadt}: 18 °C, leicht bewölkt"


@register("einwohner", "Liefert die Einwohnerzahl einer Stadt (simuliert).",
          {"type": "object", "properties": {"stadt": {"type": "string"}},
           "required": ["stadt"]})
def _einwohner(stadt):
    time.sleep(1.2)                       # simulierte Netz-Latenz
    daten = {"Wien": "1,98 Mio.", "Graz": "295.000", "Linz": "207.000"}
    return f"{stadt}: {daten.get(stadt, 'unbekannt')} Einwohner"


print("Registrierte Tools:", list(DISPATCH))

## 1 · Warum überhaupt streamen?

Ein normaler Call **blockiert**, bis die *komplette* Antwort fertig ist. Bei einer
langen Antwort starrt der User auf einen leeren Bildschirm — auch wenn das erste Wort
längst feststeht.

Mit `stream=True` bekommen wir stattdessen viele kleine **Chunks**. Jeder trägt ein
Delta in `chunk.choices[0].delta.content`. Wir drucken es sofort mit
`end="", flush=True` — der Text **wächst live**.

In [ ]:
print("— Blockierend (erst Stille, dann alles auf einmal) —")
t0 = time.time()
resp = llm([{"role": "user", "content": "Erkläre in 2 Sätzen, was ein Agentic Loop ist."}])
print(f"[{time.time()-t0:.1f}s bis zum ersten sichtbaren Zeichen]")
print(resp.choices[0].message.content)

print("\n— Streaming (Token für Token, sofort sichtbar) —")
t0 = time.time()
first = None
for chunk in stream_llm([{"role": "user", "content": "Erkläre in 2 Sätzen, was ein Agentic Loop ist."}]):
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta.content
    if delta:
        if first is None:
            first = time.time() - t0
        print(delta, end="", flush=True)
print(f"\n[erstes Zeichen bereits nach {first:.1f}s]")

👉 Inhaltlich identisch — aber die **gefühlte** Wartezeit ist eine andere Welt. Der
erste Token kommt fast sofort; der Rest tröpfelt nach. Das ist der ganze UX-Gewinn.

## 2 · Tool-Calls aus dem Stream zusammensetzen

Reiner Text ist einfach. Knifflig wird es bei **Tool-Calls**: Die kommen im Stream
**fragmentiert**. Pro `tool_call` (identifiziert über seinen `index`) liefert

- der **erste** Chunk `id` und `function.name`,
- die **folgenden** Chunks jeweils ein Stück von `function.arguments` (z. B. `{"sta`,
  dann `dt": "W`, dann `ien"}`).

Wir müssen die Stücke pro `index` **aufsammeln**, bis das JSON wieder vollständig ist.
`accumulate_stream` macht genau das: Es reicht Text-Deltas über einen Callback nach
außen (zum Live-Drucken) **und** rekonstruiert am Ende `content` + die kompletten
`tool_calls`.

In [ ]:
def accumulate_stream(stream, on_text=None):
    """Konsumiert einen Streaming-Iterator und baut die Antwort wieder zusammen.

    - on_text(delta): optionaler Callback pro Text-Token (für Live-Ausgabe)
    - Rückgabe: (content:str, tool_calls:list[dict]) — wie bei einer Nicht-Stream-Antwort
    """
    content_parts = []
    tool_calls = {}   # index -> {"id", "name", "args": [stücke...]}

    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta

        if delta.content:
            content_parts.append(delta.content)
            if on_text:
                on_text(delta.content)

        for tc in (delta.tool_calls or []):
            slot = tool_calls.setdefault(tc.index, {"id": None, "name": None, "args": []})
            if tc.id:
                slot["id"] = tc.id
            if tc.function and tc.function.name:
                slot["name"] = tc.function.name
            if tc.function and tc.function.arguments:
                slot["args"].append(tc.function.arguments)   # Stück anhängen

    # in das gewohnte tool_calls-Format gießen (args wieder zu einem String fügen)
    calls = [{"id": s["id"], "type": "function",
              "function": {"name": s["name"], "arguments": "".join(s["args"]) or "{}"}}
             for _, s in sorted(tool_calls.items())]
    return "".join(content_parts), calls


# Probe: ein Stream, der ein Tool anfordert -> sauber rekonstruiert
content, calls = accumulate_stream(
    stream_llm([{"role": "user", "content": "Wie ist das Wetter in Wien?"}], tools=TOOLS)
)
print("content:", repr(content))
print("tool_calls:", json.dumps(calls, ensure_ascii=False, indent=2))

👉 Trotz dutzender Mini-Chunks haben wir wieder einen sauberen `tool_call` mit
vollständigem `arguments`-JSON. Genau dieses Stück braucht der Loop später.

## 3 · `print`s werden zu Events

Im Grund-Notebook hat der Loop direkt ge-`print`et. Das vermischt **Logik** und
**Darstellung**. Besser: Der Loop *publiziert* neutrale, typisierte **Events** — und
*wer auch immer* zuhört, entscheidet selbst, was er damit macht (rendern, zählen,
loggen, ignorieren).

Ein Event ist bewusst minimal: ein `type`, ein `data`-Feld und die `task_id`, zu der
es gehört (wichtig, sobald mehrere Aufträge durch dieselbe Queue laufen).

In [ ]:
# Event-Typen (als simple Konstanten gehalten — kein Enum-Overhead im Vortrag)
TEXT_DELTA  = "text_delta"    # ein Stück der Antwort (Streaming-Token)
TOOL_CALL   = "tool_call"     # Agent ruft ein Tool auf
TOOL_RESULT = "tool_result"   # Ergebnis eines Tools
STEP        = "step"          # ein neuer Loop-Schritt beginnt
FINAL       = "final"         # finale Antwort steht
ERROR       = "error"         # etwas ist schiefgegangen
TASK_DONE   = "task_done"     # Auftrag komplett abgearbeitet


@dataclass
class AgentEvent:
    type: str
    task_id: int = -1
    data: Any = None


print("Event-Typen bereit:",
      [TEXT_DELTA, TOOL_CALL, TOOL_RESULT, STEP, FINAL, ERROR, TASK_DONE])

## 4 · Wer reicht die Events herum? — Auftrags-Queue + Event-Bus

Zwei einfache Bausteine aus dem Standard-Modul `queue` / `threading`:

**(a) Auftrags-Queue** (`task_queue`) — der **Input**. Aufträge landen hier; der Agent
holt sie sich als **Worker** selbst heraus (`get()` blockiert, bis etwas da ist). Ein
`None` als *Sentinel* sagt dem Worker: „Feierabend".

**(b) Event-Bus** (`EventBus`) — der **Output** als **Pub/Sub**. Jeder Consumer
`subscribe()-t` und bekommt seine **eigene** Queue. `publish(event)` legt das Event in
**jede** Subscriber-Queue. So hängen UI *und* Metrik-Zähler am **selben** Strom, ohne
sich gegenseitig die Events „wegzunehmen".

In [ ]:
class EventBus:
    """Minimaler Pub/Sub: ein publish, beliebig viele Subscriber-Queues."""

    def __init__(self):
        self._subscribers = []
        self._lock = threading.Lock()

    def subscribe(self):
        q = queue.Queue()
        with self._lock:
            self._subscribers.append(q)
        return q

    def publish(self, event):
        with self._lock:
            subs = list(self._subscribers)
        for q in subs:                 # jedes Event in JEDE Subscriber-Queue
            q.put(event)


# thread-sicheres Drucken (mehrere Consumer schreiben gleichzeitig)
_print_lock = threading.Lock()
def vprint(*a, **kw):
    with _print_lock:
        print(*a, **kw)


print("EventBus + Auftrags-Queue-Muster bereit.")

## 5 · Der Loop, der streamt — `run_agent_streaming`

Jetzt verschmelzen wir alles. Es ist **derselbe** Agentic Loop wie immer
(Modell fragen → Tools ausführen → Ergebnis anhängen → wiederholen), nur:

1. Wir rufen `stream_llm` statt `llm`.
2. Pro Text-Token publizieren wir ein `TEXT_DELTA`-Event.
3. Wir setzen den Stream mit `accumulate_stream` zusammen.
4. Für jeden Tool-Call publizieren wir `TOOL_CALL`, führen ihn über `DISPATCH` aus und
   publizieren das `TOOL_RESULT`.
5. Gibt es keine Tools mehr → `FINAL`.

Der Loop **druckt nichts** mehr selbst — er reicht nur Events an `publish` weiter.

In [ ]:
def to_assistant_dict(content, tool_calls):
    """content + tool_calls -> serialisierbares Assistant-Dict für die Historie."""
    d = {"role": "assistant", "content": content or ""}
    if tool_calls:
        d["tool_calls"] = tool_calls
    return d


def run_agent_streaming(task, publish, task_id=-1, system=None, max_steps=10):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": task})

    for step in range(1, max_steps + 1):
        publish(AgentEvent(STEP, task_id, {"step": step}))

        # 1) Modell streamen; Text-Deltas direkt als Events publizieren
        content, tool_calls = accumulate_stream(
            stream_llm(messages, tools=TOOLS or None),
            on_text=lambda d: publish(AgentEvent(TEXT_DELTA, task_id, d)),
        )
        messages.append(to_assistant_dict(content, tool_calls))

        # 2) Keine Tools mehr -> fertig
        if not tool_calls:
            publish(AgentEvent(FINAL, task_id, content))
            return content

        # 3) Alle angeforderten Tools ausführen
        for tc in tool_calls:
            name = tc["function"]["name"]
            args = json.loads(tc["function"]["arguments"] or "{}")
            publish(AgentEvent(TOOL_CALL, task_id, {"name": name, "args": args}))
            try:
                result = DISPATCH[name](**args)
            except Exception as e:
                result = f"ERROR: {e}"
                publish(AgentEvent(ERROR, task_id, {"name": name, "error": str(e)}))
            publish(AgentEvent(TOOL_RESULT, task_id, {"name": name, "result": str(result)}))
            messages.append({"role": "tool", "tool_call_id": tc["id"], "content": str(result)})

    publish(AgentEvent(FINAL, task_id, "(max_steps erreicht)"))
    return "(max_steps erreicht)"


print("run_agent_streaming bereit.")

## 6 · Alles zusammen, live — Worker + zwei Consumer

Jetzt die eigentliche Architektur:

- **`agent_worker`** (ein Thread) holt sich Aufträge aus der `task_queue` (`get()`),
  arbeitet jeden über `run_agent_streaming` ab und publiziert dabei in den Bus. Nach
  jedem Auftrag ein `TASK_DONE`. Ein `None` beendet den Worker.
- **`ui_consumer`** (ein Thread) abonniert den Bus und **rendert live**: Tokens direkt
  aneinander, Tool-Aktivität als Hinweiszeile.
- **`metrics_consumer`** (ein zweiter Thread) abonniert **denselben** Bus und zählt
  unabhängig: Tokens, Tool-Calls, Dauer pro Auftrag.

Zwei Consumer am *einen* Event-Strom — das ist der Mehrwert der Entkopplung: Der Loop
weiß nichts von UI oder Metriken; man kann Consumer hinzufügen/entfernen, ohne den
Agenten anzufassen.

In [ ]:
def agent_worker(task_queue, bus):
    """Holt Aufträge aus der Queue und arbeitet sie streamend ab. None = Stop."""
    while True:
        item = task_queue.get()
        if item is None:                 # Sentinel -> Feierabend
            task_queue.task_done()
            break
        task_id, task = item
        try:
            run_agent_streaming(task, bus.publish, task_id=task_id)
        except Exception as e:
            bus.publish(AgentEvent(ERROR, task_id, {"error": str(e)}))
        bus.publish(AgentEvent(TASK_DONE, task_id, None))
        task_queue.task_done()


def ui_consumer(q, n_tasks):
    """Rendert den Event-Strom für den Menschen (live, Token für Token)."""
    done = 0
    while done < n_tasks:
        ev = q.get()
        if ev.type == STEP:
            vprint(f"\n[#{ev.task_id} · Schritt {ev.data['step']}] ", end="", flush=True)
        elif ev.type == TEXT_DELTA:
            vprint(ev.data, end="", flush=True)
        elif ev.type == TOOL_CALL:
            vprint(f"\n   🔧 {ev.data['name']}({ev.data['args']}) …", flush=True)
        elif ev.type == TOOL_RESULT:
            vprint(f"   👁  {ev.data['result']}", flush=True)
        elif ev.type == FINAL:
            vprint(f"\n   ✓ [#{ev.task_id}] fertig.", flush=True)
        elif ev.type == TASK_DONE:
            done += 1


def metrics_consumer(q, n_tasks):
    """Zweiter, unabhängiger Consumer: zählt nur — ohne die UI zu stören."""
    stats = {}                            # task_id -> dict
    starts = {}
    done = 0
    while done < n_tasks:
        ev = q.get()
        s = stats.setdefault(ev.task_id, {"tokens": 0, "tools": 0})
        if ev.type == STEP:
            starts.setdefault(ev.task_id, time.time())
        elif ev.type == TEXT_DELTA:
            s["tokens"] += 1
        elif ev.type == TOOL_CALL:
            s["tools"] += 1
        elif ev.type == TASK_DONE:
            dur = time.time() - starts.get(ev.task_id, time.time())
            s["dauer_s"] = round(dur, 1)
            done += 1
    vprint("\n\n📊 Metriken (zweiter Consumer, lief parallel mit):")
    for tid in sorted(stats):
        s = stats[tid]
        vprint(f"   #{tid}: {s['tokens']} Token-Deltas, {s['tools']} Tool-Calls, "
               f"{s.get('dauer_s', '?')}s")

### Starten

Wir verdrahten alles: Bus anlegen, zwei Consumer abonnieren lassen, Worker + Consumer
als Threads starten, dann **Aufträge in die Queue legen**. Der Worker zieht sie selbst
heraus — wir müssen ihn nicht „aufrufen". Am Ende ein `None` als Stop-Signal.

In [ ]:
bus = EventBus()
ui_q = bus.subscribe()          # Queue für den UI-Consumer
metrics_q = bus.subscribe()     # Queue für den Metrik-Consumer
task_queue = queue.Queue()

auftraege = [
    "Wie ist das Wetter in Wien und wie viele Einwohner hat die Stadt?",
    "Liste die Dateien im aktuellen Verzeichnis auf und fasse in einem Satz zusammen.",
    "Vergleiche die Einwohnerzahl von Graz und Linz.",
]
N = len(auftraege)

# Threads starten
worker = threading.Thread(target=agent_worker, args=(task_queue, bus), daemon=True)
ui = threading.Thread(target=ui_consumer, args=(ui_q, N), daemon=True)
metrics = threading.Thread(target=metrics_consumer, args=(metrics_q, N), daemon=True)
for t in (worker, ui, metrics):
    t.start()

# Aufträge einreihen — der Worker holt sie sich selbst
for i, a in enumerate(auftraege):
    task_queue.put((i, a))
task_queue.put(None)            # Sentinel: Worker darf nach dem letzten Auftrag stoppen

# auf die Consumer warten (UI + Metriken haben jeweils N TASK_DONE gesehen)
ui.join()
metrics.join()
worker.join()
vprint("\n— alle Aufträge abgearbeitet —")

👉 Beobachte zwei Dinge: (1) Der **Text wächst live**, Tool-Schritte erscheinen,
sobald sie passieren — nicht erst am Ende. (2) Der **Metrik-Consumer** lief die ganze
Zeit **parallel** am selben Event-Strom mit und liefert am Schluss seine Zahlen — ohne
dass der Agent oder die UI etwas davon wissen mussten.

## 7 · Mitnehmen

- **Streaming ist nur ein anderer Call.** `stream=True` liefert Chunks statt einer
  fertigen Antwort. Der einzige echte Trick: **Tool-Call-Deltas pro `index`
  zusammensetzen** (`accumulate_stream`). Der Gewinn ist UX — *time-to-first-token*.
- **Events entkoppeln Logik von Darstellung.** Der Loop `publish`-t neutrale
  `AgentEvent`s und weiß nicht, wer zuhört. UI, Metriken, Logging, Audit — alles sind
  nur Consumer.
- **Zwei Queues, ein Muster.** Eine **Auftrags-Queue** macht den Agenten zum **Worker**,
  der sich Arbeit selbst holt (skaliert zu mehreren Workern). Ein **Event-Bus** (Pub/Sub)
  verteilt einen Strom an **mehrere** Consumer gleichzeitig.
- **Es bleibt derselbe Loop.** Modell fragen → Tools ausführen → anhängen → wiederholen.
  Streaming und Queue sind *drumherum* — genau wie Memory, Tools und Harness in den
  anderen Kapiteln.